In [1]:
from os import environ
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage, SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import AzureChatOpenAI
from typing_extensions import Annotated, TypedDict
import pypandoc


In [2]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    pptx_json: str | None
    markdown: str | None
    pptx_file: str | None
    intent: str | None

In [3]:
llm = AzureChatOpenAI(model="gpt4.1-mini", 
                      api_key=environ.get("AZURE_OPENAI_API_KEY"), 
                      azure_endpoint=environ.get("AZURE_OPENAI_API_BASE"), 
                      api_version=environ.get("AZURE_OPENAI_API_VERSION"), 
                      deployment_name="gpt-4.1-mini")

In [ ]:
def generate_pptx(state: State):
    """Converts markdown content to a pptx file using Pandoc."""
    output_file = "presentation.pptx"

    pypandoc.convert_text(
        state.get("markdown", ""),
        "pptx",          # formato de salida
        format="md",     # formato de entrada
        outputfile=output_file,
        extra_args=["--standalone"]  # asegura un archivo completo
    )
    state["pptx_file"] = output_file
    return {"pptx_file": output_file}
#add pptx language
#check columns and layout
#translate prompts to english
#check templates
#check images
#llm_with_tools = llm.bind_tools([generate_pptx])


In [ ]:
def chatbot(state: State):
    pptx_file = state.get("pptx_file")

    prompt = """
    You are a presentation expert assistant. You can chat normally about any topic and, if the user requests a presentation, offer your help to create it in PPTX format.
    Never say you can't create presentations. Always be positive and professional.
    """

    if pptx_file:
        prompt = """
        You are a presentation expert assistant. The requested PPTX presentation has been successfully generated.
        Congratulate the user and ask if they would like to create another presentation on a different topic or with different characteristics.
        Never say you can't create presentations. If the user requests another one, start the process without asking for additional details.
        """
    
    return {"messages" : llm.invoke([SystemMessage(content=prompt)] + state["messages"][-10:])}

In [ ]:
def json_generator(state: State):
    prompt = """
    You are a presentation planner that generates value for the user.
Your task is to create the base structure of a presentation in JSON format.

### Instructions:
1. Don't explain your reasoning.
2. Don't include text outside the JSON.
3. Make sure the JSON is valid.
4. Each slide must have:
   - "title": short title (max. 8 words).
   - "content": object that can contain one or more of the following fields:
       * "bullets": list of short phrases (2 to 5). Optional.
       * "paragraph": short paragraph (max. 120 words). Optional.
       * "images": image description (max. 1). Optional.
       * "table": optional object.
         {
           "headers": ["col1", "col2", "col3"],
           "rows": [
             ["data1", "data2", "data3"],
             ["data4", "data5", "data6"]
           ]
         }

5. Not all slides should be the same.
   Use your judgment to generate the best presentation structure according to the topic.
   Examples:
   - A slide can be just a paragraph.
   - Another can be just a table.
   - Another can be an image.
   - Or combine bullets with an image.

6. The number of slides must match exactly what the user requested.

### Output example:
{
  "slides": [
    {
      "title": "Introduction to Blockchain",
      "content": {
        "paragraph": "Blockchain is a distributed ledger technology that enables secure transactions without intermediaries."
      }
    },
    {
      "title": "Application Comparison",
      "content": {
        "bullets": ["Finance", "Logistics", "Digital Identity"],
        "table": {
          "headers": ["Sector", "Example", "Benefit"],
          "rows": [
            ["Finance", "Bitcoin", "Decentralized payments"],
            ["Logistics", "IBM Food Trust", "Product tracking"],
            ["Identity", "Digital ID", "Enhanced security"]
          ]
        }
      }
    },
    {
      "title": "Blockchain in Images",
      "content": {
        "images": [
          "Illustration of a block chained to other blocks"
        ]
      }
    }
  ]
}
### User:
    """ + f"""
### User input:
{state["messages"]}
### JSON Response:
    """
    response = llm.invoke(prompt)
    state["pptx_json"] = response.content
    return {"pptx_json": response.content}

In [ ]:
def generate_markdown(state: State):
    prompt = f"""
You are a content transformer.
You will receive a JSON that describes a slide presentation.
You must convert it to valid Markdown, using syntax that Pandoc can process to generate a PPTX file.

### Instructions:
1. Don't invent content. Use only the information given in the JSON.
2. Don't explain anything, your output must be only Markdown.
3. Each slide must start with a level 1 title: "# title".
4. Inside each slide:
   - If there's a "paragraph", write it as normal text below the title.
   - If there are "bullets", convert them to a list with dashes (- item).
   - If there are "images", for each description add a line with the format:
       `![Description](placeholder.png)`
     (use a generic filename, as real images will be generated later).
   - If there's a "table", convert it to a Markdown table with Pandoc format:
       ```
       | Header1 | Header2 | Header3 |
       |---------|---------|---------|
       | data1   | data2   | data3   |
       | data4   | data5   | data6   |
       ```
5. Separate slides with two line breaks.

### Ejemplo de salida en Markdown:

# Introduction to Blockchain
Blockchain is a distributed ledger technology that enables secure transactions without intermediaries.

# Application Comparison
- Finance
- Logistics
- Digital Identity

| Sector   | Example        | Benefit               |
|----------|----------------|----------------------|
| Finance  | Bitcoin        | Decentralized payments|
| Logistics| IBM Food Trust | Product tracking     |
| Identity | Digital ID     | Enhanced security    |

# Blockchain in Images
![Illustration of a block chained to other blocks](placeholder.jpg)
![Digital security icon with a padlock](placeholder.jpg)

### JSON de entrada:
{state["pptx_json"]}

### Tu salida (solo Markdown):

"""
    
    response = llm.invoke(prompt)
    state["markdown"] = response.content
    return {"markdown": response.content}



In [ ]:
def classify_intent(state: State):
    prompt = (
        "Is the user requesting a slide presentation (pptx, slides, presentation)? "
        "Answer only 'yes' or 'no'. Message: " + state["messages"][-1].content
    )
    response = llm.invoke(prompt)
    state["intent"] = response.content.strip().lower()
    return {"intent": state["intent"]}

In [ ]:
def edge_condition(state: State):
    if state.get("intent") == "yes" and state.get("pptx_json") is None:
        return "json"
    elif state.get("intent") == "no":
        return END
    elif state.get("pptx_file") is not None:
        return END
    return END

In [10]:
#tool_node = ToolNode(tools=[generate_pptx])
graph = StateGraph(state_schema=State)
graph.add_node("classify_intent", classify_intent)
graph.add_node("chatbot", chatbot)
graph.add_node("json", json_generator)
graph.add_node("markdown", generate_markdown)
graph.add_node("pptx", generate_pptx)
#graph.add_node("tools", tool_node)

graph.add_edge(START, "classify_intent")
graph.add_edge("classify_intent", "chatbot")
graph.add_conditional_edges("chatbot", edge_condition)
graph.add_edge("json", "markdown")
#graph.add_edge("chatbot", "tools")
graph.add_edge("markdown", "pptx")
graph.add_edge("pptx", "chatbot")
graph.add_edge("chatbot", END)


#response = graph_build.invoke({"messages": [HumanMessage(content="Generame una presentacion de 5 slides sobre energias renovables")]})
#print(response)

In [ ]:
graph_build = graph.compile()
def stream_graph(input: str):
  events = []
  for event in graph_build.stream({"messages" : [HumanMessage(content=input)]}):
    events.append(event)
    print(event)
  

while True:
      user_input = input("User: ")
      if user_input.lower() in ["quit", "exit", "q"]:
          break
      stream_graph(user_input)

## Change json input to use the user query instead of the last message, modify edge condition to search for tool call in all messages, not just the last one. 

{'classify_intent': {'intent': 'sí'}}
{'chatbot': {'messages': AIMessage(content='¡Claro! Puedo ayudarte a crear una presentación de 5 diapositivas sobre dinosaurios que incluya tablas comparativas e imágenes. Aquí te dejo la estructura propuesta para las diapositivas:\n\n1. Título: Introducción a los Dinosaurios\n2. Clasificación de Dinosaurios (tabla comparativa entre terópodos, sauropodomorfos, y ornitisquios)\n3. Características físicas (tabla comparativa de tamaño, dieta y período en que vivieron)\n4. Ejemplos de dinosaurios famosos con imágenes (Tyrannosaurus rex, Triceratops, Brachiosaurus)\n5. Importancia y legado de los dinosaurios\n\n¿Te parece bien esta estructura? Si quieres, puedo empezar a generarla y luego te enviaré el archivo PPTX para que lo descargues. ¿Quieres que incluya algún dinosaurio o dato en particular?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 89, 'total_tokens': 273, 'completion_toke

In [ ]:
graph

In [ ]:
import pypandoc

# Markdown content
markdown_content = """% Dinosaurs
% Rafael Rivas  
% September 2025

---

# What are dinosaurs?

- Reptiles that dominated Earth during the Mesozoic era  
- Lived more than 65 million years ago  
- Include terrestrial, aquatic, and flying species  

---

# Main Types

- **Saurischians**: large carnivorous and herbivorous dinosaurs  
- **Ornithischians**: herbivores with unique defenses  
- Famous examples: *T. rex*, *Triceratops*, *Velociraptor*  

---

# Extinction

- Occurred approximately 66 million years ago  
- Main hypothesis: asteroid impact in Yucatan  
- Consequences: climate change and mass extinction  

---

# Legacy

- Descendants: modern birds  
- Source of fascination in science and popular culture  
- Inspire movies, books, and scientific discoveries  

"""

# Convert Markdown → PPTX
output_file = "presentation.pptx"

pypandoc.convert_text(
    markdown_content,
    "pptx",          # output format
    format="md",     # input format
    outputfile=output_file,
    extra_args=["--standalone"]  # ensures complete file
)

print(f"✅ Presentation created: {output_file}")


RuntimeError: Pandoc died with exitcode "1" during conversion: pandoc: presentacion.pptx: withBinaryFile: permission denied (Permission denied)
